In [1]:
# 1. Установка зависимостей

%pip install "jedi>=0.16" \
    "gradio==6.17.3" \
    "transformers==4.57.6" \
    accelerate \
    bitsandbytes \
    "sentence-transformers==5.7.0" \
    "llama-index-core==0.14.24" \
    "llama-index-llms-huggingface==0.7.0" \
    "llama-index-embeddings-huggingface==0.7.0" \
    "llama-index-readers-wikipedia==0.5.0" \
    "requests==2.32.4" \
    "beautifulsoup4==4.13.5" \
    "wikipedia==1.4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 57.0 MB/s eta 0:00:00


In [2]:
# 2. Импорты

import gc
from typing import Any
from urllib.parse import urlparse

import gradio as gr
import requests
import torch

from bs4 import BeautifulSoup
from pydantic import PrivateAttr

from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from llama_index.core import (
    Document,
    PromptTemplate,
    PropertyGraphIndex,
    Settings,
)

from llama_index.core.indices.property_graph.transformations import (
    ImplicitPathExtractor,
    SimpleLLMPathExtractor,
)

from llama_index.core.llms import (
    CompletionResponse,
    CompletionResponseGen,
    CustomLLM,
    LLMMetadata,
)

from llama_index.core.llms.callbacks import llm_completion_callback
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.readers.wikipedia import WikipediaReader


# 3. Настройки

SAIGA_MODEL = "IlyaGusev/saiga_mistral_7b_merged"
FRED_MODEL = "ai-forever/FRED-T5-1.7B"

EMBED_MODEL = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

NO_INFORMATION_ANSWER = (
    "Извините, я не нашел информации "
    "по вашему вопросу в базе знаний."
)


# 4. RAG prompt

QA_PROMPT = PromptTemplate(
    """
Ниже приведён контекст, полученный из графа знаний.

---------------------
{context_str}
---------------------

Ответь на вопрос пользователя, используя только информацию
из приведённого контекста.

Правила:
1. Не используй внешние знания.
2. Не придумывай факты.
3. Если информации недостаточно, ответь точно:
Извините, я не нашел информации по вашему вопросу в базе знаний.
4. Отвечай на русском языке.
5. Дай понятный и содержательный ответ.

Вопрос:
{query_str}

Ответ:
""".strip()
)


# 5. Saiga LLM

class SaigaLLM(CustomLLM):
    context_window: int = 3900
    num_output: int = 512
    model_name: str = SAIGA_MODEL

    _model: Any = PrivateAttr()
    _tokenizer: Any = PrivateAttr()

    def __init__(
        self,
        model,
        tokenizer,
        context_window=3900,
        num_output=512,
        **kwargs,
    ):
        super().__init__(
            context_window=context_window,
            num_output=num_output,
            model_name=SAIGA_MODEL,
            **kwargs,
        )

        self._model = model
        self._tokenizer = tokenizer

    @property
    def metadata(self) -> LLMMetadata:
        return LLMMetadata(
            context_window=self.context_window,
            num_output=self.num_output,
            model_name=self.model_name,
            is_chat_model=False,
        )

    def prepare_prompt(self, prompt: str) -> str:
        return (
            "<s>system\n"
            "Ты — русскоязычный ассистент. "
            "Строго следуй инструкции пользователя."
            "</s>\n"
            "<s>user\n"
            f"{prompt}"
            "</s>\n"
            "<s>bot\n"
        )

    @llm_completion_callback()
    def complete(
        self,
        prompt: str,
        formatted: bool = False,
        **kwargs: Any,
    ) -> CompletionResponse:
        full_prompt = self.prepare_prompt(prompt)

        max_input_length = self.context_window - self.num_output

        inputs = self._tokenizer(
            full_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_input_length,
        )

        device = next(self._model.parameters()).device

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        input_length = inputs["input_ids"].shape[1]

        with torch.inference_mode():
            output_ids = self._model.generate(
                **inputs,
                max_new_tokens=self.num_output,
                do_sample=False,
                repetition_penalty=1.1,
                eos_token_id=self._tokenizer.eos_token_id,
                pad_token_id=self._tokenizer.pad_token_id,
            )

        generated_ids = output_ids[0][input_length:]

        text = self._tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        ).strip()

        return CompletionResponse(text=text)

    @llm_completion_callback()
    def stream_complete(
        self,
        prompt: str,
        formatted: bool = False,
        **kwargs: Any,
    ) -> CompletionResponseGen:
        response = self.complete(
            prompt,
            formatted=formatted,
            **kwargs,
        )

        yield CompletionResponse(
            text=response.text,
            delta=response.text,
        )


# 6. FRED-T5 LLM

class FredT5LLM(CustomLLM):
    context_window: int = 2048
    num_output: int = 384
    model_name: str = FRED_MODEL

    _model: Any = PrivateAttr()
    _tokenizer: Any = PrivateAttr()

    def __init__(
        self,
        model,
        tokenizer,
        context_window=2048,
        num_output=384,
        **kwargs,
    ):
        super().__init__(
            context_window=context_window,
            num_output=num_output,
            model_name=FRED_MODEL,
            **kwargs,
        )

        self._model = model
        self._tokenizer = tokenizer

    @property
    def metadata(self) -> LLMMetadata:
        return LLMMetadata(
            context_window=self.context_window,
            num_output=self.num_output,
            model_name=self.model_name,
            is_chat_model=False,
        )

    def prepare_prompt(self, prompt: str) -> str:
        prompt = prompt.strip()

        if not prompt.startswith("<LM>"):
            prompt = "<LM>" + prompt

        return prompt

    @llm_completion_callback()
    def complete(
        self,
        prompt: str,
        formatted: bool = False,
        **kwargs: Any,
    ) -> CompletionResponse:
        full_prompt = self.prepare_prompt(prompt)

        inputs = self._tokenizer(
            full_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.context_window,
        )

        device = next(self._model.parameters()).device

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.inference_mode():
            output_ids = self._model.generate(
                **inputs,
                max_new_tokens=self.num_output,
                do_sample=False,
                eos_token_id=self._tokenizer.eos_token_id,
                pad_token_id=self._tokenizer.pad_token_id,
            )

        text = self._tokenizer.decode(
            output_ids[0],
            skip_special_tokens=True,
        ).strip()

        return CompletionResponse(text=text)

    @llm_completion_callback()
    def stream_complete(
        self,
        prompt: str,
        formatted: bool = False,
        **kwargs: Any,
    ) -> CompletionResponseGen:
        response = self.complete(
            prompt,
            formatted=formatted,
            **kwargs,
        )

        yield CompletionResponse(
            text=response.text,
            delta=response.text,
        )


# 7. Загрузка моделей

class ModelLoader:
    @staticmethod
    def require_cuda():
        if not torch.cuda.is_available():
            raise RuntimeError(
                "CUDA GPU не найден. "
                "В Google Colab включите GPU."
            )

    @staticmethod
    def quantization_config():
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    @staticmethod
    def load_saiga():
        ModelLoader.require_cuda()

        print("Загрузка Saiga Mistral 7B...")

        tokenizer = AutoTokenizer.from_pretrained(
            SAIGA_MODEL,
            use_fast=False,
        )

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            SAIGA_MODEL,
            quantization_config=ModelLoader.quantization_config(),
            torch_dtype=torch.float16,
            device_map="auto",
        )

        model.eval()

        return SaigaLLM(
            model=model,
            tokenizer=tokenizer,
        )

    @staticmethod
    def load_fred():
        ModelLoader.require_cuda()

        print("Загрузка FRED-T5 1.7B...")

        tokenizer = AutoTokenizer.from_pretrained(
            FRED_MODEL
        )

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForSeq2SeqLM.from_pretrained(
            FRED_MODEL,
            quantization_config=ModelLoader.quantization_config(),
            torch_dtype=torch.float16,
            device_map="auto",
        )

        model.eval()

        return FredT5LLM(
            model=model,
            tokenizer=tokenizer,
        )


# 8. Загрузка данных

class DataLoader:
    @staticmethod
    def load_from_wikipedia(topic: str):
        topic = topic.strip()

        if not topic:
            raise ValueError(
                "Введите тему Википедии."
            )

        print(
            f"Загрузка Википедии: {topic}"
        )

        reader = WikipediaReader()

        documents = reader.load_data(
            pages=[topic],
            lang_prefix="ru",
        )

        if not documents:
            raise ValueError(
                "Википедия не вернула документы."
            )

        return documents

    @staticmethod
    def validate_url(url: str):
        url = url.strip()
        parsed = urlparse(url)

        if parsed.scheme not in (
            "http",
            "https",
        ):
            raise ValueError(
                f"Некорректный URL: {url}"
            )

        if not parsed.netloc:
            raise ValueError(
                f"Некорректный URL: {url}"
            )

        return url

    @staticmethod
    def load_from_web(urls):
        if not urls:
            raise ValueError(
                "Укажите хотя бы один URL."
            )

        headers = {
            "User-Agent": (
                "Mozilla/5.0 "
                "(Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 "
                "Chrome/131 Safari/537.36"
            )
        }

        documents = []

        for url in urls:
            url = DataLoader.validate_url(url)

            print(
                f"Загрузка страницы: {url}"
            )

            response = requests.get(
                url,
                headers=headers,
                timeout=30,
            )

            response.raise_for_status()

            soup = BeautifulSoup(
                response.text,
                "html.parser",
            )

            for tag in soup(
                [
                    "script",
                    "style",
                    "noscript",
                    "nav",
                    "header",
                    "footer",
                    "form",
                    "svg",
                ]
            ):
                tag.decompose()

            text = soup.get_text(
                separator="\n",
                strip=True,
            )

            if not text:
                continue

            text = text[:200000]

            documents.append(
                Document(
                    text=text,
                    metadata={
                        "source": url
                    },
                )
            )

        if not documents:
            raise ValueError(
                "Не удалось получить текст "
                "с указанных страниц."
            )

        return documents

    @staticmethod
    def load_from_text(text: str):
        text = text.strip()

        if not text:
            raise ValueError(
                "Введите текст."
            )

        return [
            Document(
                text=text,
                metadata={
                    "source": "manual_text"
                },
            )
        ]


# 9. Graph RAG bot

class GraphRAGBot:
    def __init__(self):
        self.model_type = "saiga"
        self.loaded_model_type = None

        self.llm = None
        self.embed_model = None

        self.documents = None
        self.index = None
        self.query_engine = None

        self.initialized = False

    def release_model(self):
        self.query_engine = None
        self.index = None
        self.llm = None

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        self.loaded_model_type = None
        self.initialized = False

    def initialize(self, model_type):
        if model_type not in (
            "saiga",
            "fred",
        ):
            return (
                "Ошибка: неизвестная модель."
            )

        if (
            self.initialized
            and self.llm is not None
            and self.loaded_model_type == model_type
        ):
            return (
                f"Модель '{model_type}' уже загружена."
            )

        try:
            if self.llm is not None:
                self.release_model()

            self.model_type = model_type

            if model_type == "saiga":
                self.llm = ModelLoader.load_saiga()
            else:
                self.llm = ModelLoader.load_fred()

            if self.embed_model is None:
                print(
                    "Загрузка embedding-модели..."
                )

                self.embed_model = HuggingFaceEmbedding(
                    model_name=EMBED_MODEL,
                    device="cpu",
                )

            Settings.llm = self.llm
            Settings.embed_model = self.embed_model
            Settings.chunk_size = 512
            Settings.chunk_overlap = 50

            self.loaded_model_type = model_type
            self.initialized = True

            return (
                f"Модель '{model_type}' "
                "успешно загружена."
            )

        except Exception as exc:
            self.llm = None
            self.loaded_model_type = None
            self.initialized = False

            return (
                "Ошибка загрузки модели:\n"
                f"{type(exc).__name__}: {exc}"
            )

    def load_data(
        self,
        source_type,
        wiki_topic,
        web_urls,
        text_data,
    ):
        try:
            if source_type == "wikipedia":
                self.documents = (
                    DataLoader.load_from_wikipedia(
                        wiki_topic
                    )
                )

            elif source_type == "web":
                urls = [
                    url.strip()
                    for url in web_urls.split(",")
                    if url.strip()
                ]

                self.documents = (
                    DataLoader.load_from_web(
                        urls
                    )
                )

            elif source_type == "text":
                self.documents = (
                    DataLoader.load_from_text(
                        text_data
                    )
                )

            else:
                return (
                    "Ошибка: неизвестный "
                    "источник данных."
                )

            self.index = None
            self.query_engine = None

            return (
                "Данные успешно загружены.\n"
                f"Документов: "
                f"{len(self.documents)}"
            )

        except Exception as exc:
            return (
                "Ошибка загрузки данных:\n"
                f"{type(exc).__name__}: {exc}"
            )

    def build_graph(self):
        if not self.initialized:
            return (
                "Сначала инициализируйте модель."
            )

        if not self.documents:
            return (
                "Сначала загрузите данные."
            )

        try:
            print(
                "Построение графа знаний..."
            )

            Settings.llm = self.llm
            Settings.embed_model = self.embed_model

            kg_extractors = [
                SimpleLLMPathExtractor(
                    llm=self.llm,
                    max_paths_per_chunk=10,
                    num_workers=1,
                ),
                ImplicitPathExtractor(),
            ]

            self.index = PropertyGraphIndex.from_documents(
                self.documents,
                llm=self.llm,
                embed_model=self.embed_model,
                kg_extractors=kg_extractors,
                embed_kg_nodes=True,
                use_async=False,
                show_progress=True,
            )

            retriever = self.index.as_retriever(
                include_text=True,
                similarity_top_k=5,
                path_depth=1,
                use_async=False,
            )

            self.query_engine = (
                RetrieverQueryEngine.from_args(
                    retriever=retriever,
                    llm=self.llm,
                    response_mode="compact",
                    text_qa_template=QA_PROMPT,
                )
            )

            return (
                "Граф знаний успешно построен."
            )

        except Exception as exc:
            self.index = None
            self.query_engine = None

            return (
                "Ошибка построения графа:\n"
                f"{type(exc).__name__}: {exc}"
            )

    def ask(self, question: str):
        question = (
            question or ""
        ).strip()

        if not question:
            return (
                "Введите вопрос."
            )

        if self.query_engine is None:
            return (
                "Сначала загрузите данные "
                "и постройте граф знаний."
            )

        try:
            response = self.query_engine.query(
                question
            )

            answer = str(
                response
            ).strip()

            if not answer:
                return NO_INFORMATION_ANSWER

            return answer

        except Exception as exc:
            return (
                "Ошибка обработки вопроса:\n"
                f"{type(exc).__name__}: {exc}"
            )

    def status(self):
        if torch.cuda.is_available():
            gpu_name = (
                torch.cuda.get_device_name(0)
            )

            total_memory = (
                torch.cuda
                .get_device_properties(0)
                .total_memory
                / 1024 ** 3
            )

            gpu_text = (
                f"{gpu_name} "
                f"({total_memory:.1f} GB)"
            )

        else:
            gpu_text = (
                "CUDA недоступна"
            )

        return (
            f"Выбрана модель: "
            f"{self.model_type}\n"

            f"Загружена модель: "
            f"{self.loaded_model_type or 'Нет'}\n"

            f"Инициализирована: "
            f"{'Да' if self.initialized else 'Нет'}\n"

            f"Документов: "
            f"{len(self.documents) if self.documents else 0}\n"

            f"Граф: "
            f"{'Построен' if self.index else 'Нет'}\n"

            f"GPU: {gpu_text}"
        )


# 10. Функции Gradio

bot = GraphRAGBot()


def ui_initialize(model_type):
    result = bot.initialize(
        model_type
    )

    return (
        result,
        bot.status(),
    )


def ui_load_data(
    source_type,
    wiki_topic,
    web_urls,
    text_data,
):
    result = bot.load_data(
        source_type,
        wiki_topic,
        web_urls,
        text_data,
    )

    return (
        result,
        bot.status(),
    )


def ui_build_graph():
    result = bot.build_graph()

    return (
        result,
        bot.status(),
    )


def ui_chat(
    question,
    history,
):
    question = (
        question or ""
    ).strip()

    history = list(
        history or []
    )

    if not question:
        return (
            history,
            "",
        )

    answer = bot.ask(
        question
    )

    history.append(
        {
            "role": "user",
            "content": question,
        }
    )

    history.append(
        {
            "role": "assistant",
            "content": answer,
        }
    )

    return (
        history,
        "",
    )


def ui_clear():
    return (
        [],
        "",
    )


def update_source(source_type):
    return (
        gr.update(
            visible=(
                source_type == "wikipedia"
            )
        ),
        gr.update(
            visible=(
                source_type == "web"
            )
        ),
        gr.update(
            visible=(
                source_type == "text"
            )
        ),
    )


# 11. Интерфейс

def create_interface():
    with gr.Blocks(
        title="Graph RAG"
    ) as interface:

        gr.Markdown(
            """
# Graph RAG чат-бот

**Модели:** Saiga Mistral 7B / FRED-T5 1.7B

**Источники:** Википедия / веб-страницы / произвольный текст

1. Выберите модель.
2. Инициализируйте модель.
3. Загрузите данные.
4. Постройте граф знаний.
5. Задавайте вопросы.
"""
        )

        with gr.Row():

            with gr.Column(
                scale=1
            ):
                model_choice = gr.Dropdown(
                    choices=[
                        "saiga",
                        "fred",
                    ],
                    value="saiga",
                    label="Модель",
                )

                init_button = gr.Button(
                    "Инициализировать модель",
                    variant="primary",
                )

                source_type = gr.Dropdown(
                    choices=[
                        "wikipedia",
                        "web",
                        "text",
                    ],
                    value="wikipedia",
                    label="Источник данных",
                )

                with gr.Group(
                    visible=True
                ) as wikipedia_group:
                    wiki_topic = gr.Textbox(
                        label="Тема Википедии",
                        value="Искусственный интеллект",
                    )

                with gr.Group(
                    visible=False
                ) as web_group:
                    web_urls = gr.Textbox(
                        label="URL через запятую",
                        placeholder="https://example.com",
                        lines=3,
                    )

                with gr.Group(
                    visible=False
                ) as text_group:
                    text_data = gr.Textbox(
                        label="Произвольный текст",
                        placeholder="Введите текст...",
                        lines=8,
                    )

                load_button = gr.Button(
                    "Загрузить данные"
                )

                build_button = gr.Button(
                    "Построить граф знаний",
                    variant="primary",
                )

                operation_output = gr.Textbox(
                    label="Результат операции",
                    lines=5,
                    interactive=False,
                )

                status_output = gr.Textbox(
                    label="Статус",
                    value=bot.status(),
                    lines=7,
                    interactive=False,
                )

            with gr.Column(
                scale=2
            ):
                chatbot = gr.Chatbot(
                    label="Диалог",
                    height=550,
                )

                question = gr.Textbox(
                    label="Вопрос",
                    placeholder=(
                        "Задайте вопрос "
                        "по базе знаний..."
                    ),
                    lines=2,
                )

                with gr.Row():
                    ask_button = gr.Button(
                        "Отправить",
                        variant="primary",
                    )

                    clear_button = gr.Button(
                        "Очистить"
                    )

        source_type.change(
            fn=update_source,
            inputs=[
                source_type
            ],
            outputs=[
                wikipedia_group,
                web_group,
                text_group,
            ],
        )

        init_button.click(
            fn=ui_initialize,
            inputs=[
                model_choice
            ],
            outputs=[
                operation_output,
                status_output,
            ],
        )

        load_button.click(
            fn=ui_load_data,
            inputs=[
                source_type,
                wiki_topic,
                web_urls,
                text_data,
            ],
            outputs=[
                operation_output,
                status_output,
            ],
        )

        build_button.click(
            fn=ui_build_graph,
            inputs=[],
            outputs=[
                operation_output,
                status_output,
            ],
        )

        ask_button.click(
            fn=ui_chat,
            inputs=[
                question,
                chatbot,
            ],
            outputs=[
                chatbot,
                question,
            ],
        )

        question.submit(
            fn=ui_chat,
            inputs=[
                question,
                chatbot,
            ],
            outputs=[
                chatbot,
                question,
            ],
        )

        clear_button.click(
            fn=ui_clear,
            inputs=[],
            outputs=[
                chatbot,
                question,
            ],
        )

    return interface


# 12. Запуск

interface = create_interface()

interface.queue(
    default_concurrency_limit=1
)

interface.launch(
    share=True,
    show_error=True,
    theme=gr.themes.Soft(),
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3999ad8d272e63a19e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
